# Structural Tracking with SDC Kernel

This notebook demonstrates structural tracking with the SDC (Sequential Dataflow Consistency) kernel.

The SDC kernel enforces dataflow consistency across cells. With structural tracking,
it also detects when structural changes (column/row additions) affect earlier cells
that read structural attributes.

**Note**: This notebook should be run with the `ferret_sdc_kernel` kernel.

In [ ]:
import pandas as pd

# Check current structural tracking mode
%structural_tracking

In [ ]:
# Set to ENFORCE mode for demonstration
%structural_tracking enforce

## Scenario: Multi-Cell Structural Dependency

Cell A: Creates a DataFrame
Cell B: Reads the column structure  
Cell C: Adds a column

With SDC + structural tracking:
- Cell B depends on the column structure
- Cell C's modification makes Cell B stale

In [ ]:
# Cell A: Create DataFrame
df = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Gizmo'],
    'price': [10.00, 25.00, 15.00]
})
df

In [ ]:
# Cell B: Read column structure
print(f"DataFrame has {len(df.columns)} columns: {list(df.columns)}")
print(f"DataFrame has {len(df)} rows")

In [ ]:
# Cell C: Add a column
# In ENFORCE mode with SDC, this should be detected as affecting Cell B
df['quantity'] = [100, 50, 75]
df

## Checking Staleness

After Cell C runs, Cell B should be marked as stale because:
- Cell B read `df.columns` and `len(df)` (structural reads)
- Cell C added a column, changing the column structure

Use `%sdc_stale` to see which cells are stale.

In [ ]:
%sdc_stale

## Mode Comparison

- **OFF**: No structural tracking. Cell B wouldn't be marked stale.
- **WARN**: Cell B marked stale with a warning message.
- **ENFORCE**: Cell B marked stale, and backward mutations blocked.

In [ ]:
# Reset to default
%structural_tracking warn